In [ ]:
import requests
import json
import pandas as pd
from utils_adv import load_api_tokens
from time import sleep
from datetime import datetime, timedelta
import gspread
import psycopg2
from psycopg2 import extras
from dotenv import load_dotenv
import os
from pandas import json_normalize
import json
import time



yesterday = datetime.now() - timedelta(days=1)
date_from = date_to = yesterday.strftime("%Y-%m-%d")
date_range = [date_from]

camps = []
campaign_statuses = [7, 8, 9, 11]


account = load_api_tokens()['Вектор']
api_token = load_api_tokens()[account]
camps = []
url = 'https://advert-api.wildberries.ru/adv/v1/promotion/adverts'
headers = {'Authorization': api_token}

# Таймер для отслеживания частоты запросов
request_count = 0
start_time = time.time()

for status_id in campaign_statuses:
    # Проверка на лимит запросов (5 запросов в секунду)
    if request_count >= 5:
        elapsed_time = time.time() - start_time
        if elapsed_time < 1:
            time.sleep(1 - elapsed_time)  # Пауза до полного заполнения секунды
        request_count = 0
        start_time = time.time()

    params = {'status': status_id}
    if type_advert:
        params['type'] = type_advert
    if ids:
        params['id'] = ids

    try:
        res = requests.post(url, headers=headers, params=params)
        res.raise_for_status()
        response_data = res.json()

        if response_data:
            # Добавляем информацию о кабинете в данные
            for item in response_data:
                item['account'] = account
            camps.extend(response_data)

    except Exception as e:
        print(f"Ошибка при запросе со статусом {status_id}: {e}")

    request_count += 1  # Увеличиваем счетчик запросов

    return camps



Ошибка при запросе со статусом 8: Expecting value: line 1 column 1 (char 0)
Ошибка при запросе со статусом 8: Expecting value: line 1 column 1 (char 0)
Ошибка при запросе со статусом 8: Expecting value: line 1 column 1 (char 0)
Ошибка при запросе со статусом 8: Expecting value: line 1 column 1 (char 0)
Ошибка при запросе со статусом 8: Expecting value: line 1 column 1 (char 0)
Ошибка при запросе со статусом 8: Expecting value: line 1 column 1 (char 0)
Ошибка при запросе со статусом 8: Expecting value: line 1 column 1 (char 0)
Ошибка при запросе со статусом 8: Expecting value: line 1 column 1 (char 0)
Ошибка при запросе со статусом 7: Expecting value: line 1 column 1 (char 0)
Ошибка при запросе со статусом 8: Expecting value: line 1 column 1 (char 0)
Ошибка при запросе со статусом 7: Expecting value: line 1 column 1 (char 0)
Ошибка при запросе со статусом 8: Expecting value: line 1 column 1 (char 0)
Ошибка при запросе со статусом 9: Expecting value: line 1 column 1 (char 0)
464
525
419


In [3]:
df_merge.head()

,advert_id,nm_id,type,account
0,9637856,164380026,8,Лопатина
1,9864067,166221596,8,Лопатина
2,9985343,162395148,8,Тоноян
3,10127196,63094114,8,Тоноян
4,10395970,165083068,8,Тоноян


In [ ]:
load_dotenv()
user = os.getenv('USER_2')
name = os.getenv('NAME_2')
password = os.getenv('PASSWORD_2')
host = os.getenv('HOST_2')
port = os.getenv('PORT_2')
dialect = 'postgresql'

# Устанавливаем таймаут в секундах
CONNECTION_TIMEOUT = 10  # Таймаут для подключения
QUERY_TIMEOUT = 5  # Таймаут для выполнения запроса (5 минут)
# Устанавливаем подключение к базе данных через psycopg2
try:
    # Создание подключения с таймаутом
    conn = psycopg2.connect(
        dbname=name,
        user=user,
        password=password,
        host=host,
        port=port,
        connect_timeout=CONNECTION_TIMEOUT  # Таймаут для установки соединения
    )
    conn.autocommit = False  # Отключаем автокоммит
    print("Соединение с базой данных установлено.")

    # Загрузка данных из DataFrame в базу данных
    table_name = 'advert_general_info'
    # Создание курсора
    cur = conn.cursor()

    # Устанавливаем таймаут для выполнения запроса
    cur.execute(f"SET statement_timeout = {QUERY_TIMEOUT * 100};")  # В миллисекундах

    # Проверяем, существует ли таблица orders, и создаем ее, если нет
    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS advert_general_info (
            advert_id BIGINT PRIMARY KEY,
            nm_id BIGINT,
            type INT,
            account VARCHAR(255)
        );""")
    print(f"Таблица {table_name} создана или уже существует.")

    columns = [
        'advert_id', 'nm_id','type', 'account'
    ]

    # Используем execute_values для оптимизации вставки
    tuples = [tuple(x) for x in df_merge.to_numpy()]
    insert_query = f"""
    INSERT INTO {table_name} ({', '.join(columns)})
    VALUES %s
    ON CONFLICT (advert_id) DO NOTHING;"""
    extras.execute_values(cur, insert_query, tuples)

    # Сохраняем изменения
    conn.commit()
    print("Данные успешно добавлены в БД.")

except psycopg2.OperationalError as e:
    print(f"Ошибка подключения к БД: {e}")
except psycopg2.DatabaseError as e:
    print(f"Ошибка при загрузке данных в БД: {e}")
    if conn:
        conn.rollback()  # Откат транзакции при ошибке
finally:
    # Закрытие соединения
    if conn:
        conn.close()
        print("Соединение с базой данных закрыто.")